<a href="https://colab.research.google.com/github/DeepthiManthapuram/Deep_Learning/blob/main/GPT_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Data Preparation

### Objective
Prepare text data for GPT training.

- Load dataset
- Convert to lowercase
- Remove unwanted symbols
- Tokenize text
- Build vocabulary
- Create input-output sequences

Example:
Input: 'the cat chased'

In [8]:
import re
from tensorflow.keras.preprocessing. text import Tokenizer

texts = [
"The cat chased the mouse.",
"The dog barked loudly.",
"Machine learning is powerful.",
"Deep learning uses neural networks."
]

texts=[re.sub(r'[^a-zA-Z\s]','',t.lower()) for t in texts]

tokenizer=Tokenizer()
tokenizer.fit_on_texts(texts)

word_index=tokenizer.word_index
vocab_size=len(word_index)+1

sequences=[]

for text in texts:
  tokens=tokenizer.texts_to_sequences([text])[0]
  for i in range(1,len(tokens)):
    sequences.append((tokens[:i], tokens[i]))

print("Vocabulary Size:",vocab_size)
print("Sample Sequence:", sequences[0])

Vocabulary Size: 16
Sample Sequence: ([1], 3)


# Task 2: Token Embeddings
Objective
Convert token IDs into dense vectors.
Example:
cat → [0.12,0.55,0.34]
dog → [0.21,0.77,0.61]

In [11]:
import tensorflow as tf

embedding_dim=32
embedding_layer=tf.keras.layers. Embedding(vocab_size,embedding_dim)

sample_input=tf.constant([[1,2,3]])
embedded=embedding_layer(sample_input)
print("Embeddings",embedded)
print("Embedding Shape", embedded.shape)

Embeddings tf.Tensor(
[[[-1.33749470e-02  3.45859267e-02 -2.85028350e-02  4.50315326e-03
    3.37356813e-02  1.84004344e-02 -1.29581094e-02 -1.96262244e-02
   -1.80698037e-02  2.82698609e-02 -3.98421995e-02  4.66148369e-02
    3.45614217e-02 -2.34439019e-02 -7.07287714e-03 -2.17761155e-02
    2.56286524e-02 -3.60393897e-02  1.55193545e-02 -3.10061127e-03
   -3.38055491e-02 -1.55376419e-02 -3.14374790e-02  1.04836114e-02
   -8.05599615e-03  1.24360435e-02 -3.21928151e-02  2.65395381e-02
   -4.02418599e-02  1.92578882e-03 -2.52322797e-02 -2.63503324e-02]
  [ 3.66976298e-02  5.28723001e-03  1.74332373e-02  3.82201113e-02
   -1.58796087e-02 -2.16954108e-02 -1.08921155e-02 -1.56510249e-02
   -2.80456673e-02 -1.73599608e-02  4.15179767e-02  3.85370515e-02
    1.07295886e-02  4.15838696e-02  1.13702640e-02 -4.80852835e-02
   -4.56272848e-02 -6.07176870e-03  1.44556202e-02  4.40833904e-02
   -2.45252848e-02  4.64326777e-02 -4.41598669e-02 -2.62038354e-02
   -3.00644282e-02  4.95382883e-02  6.4

# Task 3: Positional Encoding
Objective
Add positional information.
Example:
the → position 0
cat → position 1
chased → position 2

In [15]:
import numpy as np

def positional_encoding(max_len, d_model):
  pos = np.arange(max_len)[:, np.newaxis]
  i = np.arange(d_model)[np.newaxis, :]

  angle_rates = 1 / np.power(10000, (2*(i//2))/np.float32(d_model))
  angles = pos * angle_rates

  pe = np.zeros((max_len, d_model))
  pe[:, 0 :: 2] = np.sin(angles[:, 0 :: 2])
  pe[:, 1 :: 2] = np.cos(angles[:, 1 :: 2])

  return pe

pe = positional_encoding(10, 32)
print(pe.shape)

(10, 32)


# Task 4: Masked Self Attention
Objective
Implement causal attention.
Example:
Current token:
cat
Allowed:
the
cat
Not Allowed:
chased
mouse
Deliverables
Display:
Attention Mask
Attention Scores
Attention Weights

In [16]:
import tensorflow as tf

seq_len=5

mask=1-tf.linalg.band_part(tf.ones((seq_len,seq_len)),-1,0)
print("Attention Mask", mask. numpy())

Attention Mask [[0. 1. 1. 1. 1.]
 [0. 0. 1. 1. 1.]
 [0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0.]]


# Task 5: Multi Head Attention
Implement:
4 Attention Heads
Analyze:
Head 1 → Grammar
Head 2 → Context
Head 3 → Long Dependencies
Head 4 → Semantics


In [17]:
mha=tf.keras.layers.MultiHeadAttention(
  num_heads=4,
  key_dim=32
)

x=tf.random.normal((2,5,32))
attn_output=mha(x,x,attention_mask=mask)
print("Attention Output", attn_output.shape)

Attention Output (2, 5, 32)


# Task 6: GPT Decoder Block
Architecture:
Masked Multi Head Attention
        ↓
Add & Normalize
        ↓
Feed Forward Network
        ↓
Add & Normalize
Implement from scratch.

In [20]:
class GPTDecoderBlock(tf.keras.layers.Layer):
  def __init__(self, d_model, num_heads, dff):
    super().__init__()

    self.mha = tf.keras.layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model
    )

    self.ffn = tf.keras.Sequential([
        tf.keras.layers.Dense(dff, activation='relu'),
        tf.keras.layers.Dense(d_model)
    ])

    self.norm1 = tf.keras.layers.LayerNormalization()
    self.norm2 = tf.keras.layers.LayerNormalization()

  def call(self, x):
    attn = self.mha(x, x, use_causal_mask=True)
    x = self.norm1(x + attn)

    ffn_out = self.ffn(x)
    x = self.norm2(x + ffn_out)

    return x

# Task 7: GPT Model
Architecture:
Input Tokens
      ↓
Embedding
      ↓
Position Encoding
      ↓
Decoder Block
      ↓
Decoder Block
      ↓
Linear
      ↓
Softmax

In [24]:
inputs = tf.keras.Input(shape=(None,))

x = tf.keras.layers.Embedding(vocab_size, 32)(inputs)

decoder = GPTDecoderBlock(32,4,64)
x = decoder(x)
x = decoder(x)

outputs = tf.keras.layers.Dense(vocab_size, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 32)  │        512 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gpt_decoder_block   │ (None, None, 32)  │     21,120 │ embedding_6[0][0… │
│ (GPTDecoderBlock)   │                   │            │ gpt_decoder_bloc… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 16)  │        528 │ gpt_decoder_bloc… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 22,160 (86.56 KB)

 Trainable params: 22,160 (86.56 KB)

 Non-trainable params: 0 (0.00 B)

## Task 8: Next Token Prediction
Input:
The cat chased
Prediction:
the
Input:
The cat chased the
Prediction:
mouse

In [29]:
sentence = "Deep learning uses"

tokens = tokenizer.texts_to_sequences([sentence])[0]

import numpy as np

# Prepare input for the model
# The model expects a batch, so we add an extra dimension
input_tokens = np.array([tokens])

# Get model prediction for the next word
predictions = model.predict(input_tokens)

# The output of the model is a probability distribution over the vocabulary for each token in the input sequence.
# we are interested in the prediction for the "last" token in the sequence to predict the "next" word.
#So, we take the probabilities for the last input token and find the token with the highest probability.
predicted_token_id = np.argmax(predictions[0, -1, :])

# Convert the token ID back to a word
# We need to inverse the word_index to get word from index
reverse_word_index = dict([(value, key) for (key, value) in tokenizer.word_index.items()])
predicted_word = reverse_word_index.get(predicted_token_id, '<unk>') # Use <unk> for unknown words

print("predicted Next Word:", predicted_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
predicted Next Word: the


# Task 9: Text Generation
Input Prompt:
Machine learning
Generation:
Machine learning is
Machine learning is powerful
Machine learning is powerful because
...
Explain:
Autoregressive Generation

In [30]:
def generate_text(start_prompt, model, tokenizer, num_generate_words, max_sequence_length=None):
  generated_text = start_prompt
  current_sequence = start_prompt

  for _ in range(num_generate_words):
    # Tokenize the current sequence
    token_list = tokenizer.texts_to_sequences([current_sequence])[0]

    # If max_sequence_length is specified, truncate or pad the input sequence
    if max_sequence_length is not None:
        if len(token_list) > max_sequence_length:
            token_list = token_list[-max_sequence_length:] # Truncate from the left
        else:
            # Pad with zeros if less than max_sequence_length
            token_list = tf.keras.preprocessing.sequence.pad_sequences(
                [token_list],
                maxlen=max_sequence_length,
                padding='pre' # Pad at the beginning to maintain causal context
            )[0]

    # Prepare input for the model (add batch dimension)
    input_for_prediction = np.array([token_list])

    # Get model prediction for the next word
    predictions = model.predict(input_for_prediction, verbose=0) # verbose=0 to suppress output

    # The prediction is for the last token in the input sequence. We need to get the probabilities for the *next* token.
    predicted_token_id = np.argmax(predictions[0, -1, :])

    # Convert the token ID back to a word
    reverse_word_index = dict([(value, key) for (key, value) in tokenizer.word_index.items()])
    predicted_word = reverse_word_index.get(predicted_token_id, '') # Use empty string for unknown to avoid errors

    # Append the predicted word to the generated text
    generated_text += " " + predicted_word
    current_sequence += " " + predicted_word

  return generated_text

In [31]:
prompt = "machine learning"
num_words_to_generate = 5

# We need to determine the maximum sequence length based on our training data or model design.
# For now, let's use a arbitrary length (e.g., 10), or infer from sequences if possible.
# For this simple example, the model expects an input length based on the `sequences` created earlier.
# The `max_sequence_length` in this case would be the length of the longest input sequence in `sequences`.
# Let's find the max length from our 'sequences' data preparation step.

if len(sequences) > 0:
    max_seq_len_for_padding = max(len(s[0]) for s in sequences) + 1 # +1 for the target token
else:
    max_seq_len_for_padding = 10 # A reasonable default if no sequences are found

generated_output = generate_text(prompt, model, tokenizer, num_words_to_generate, max_sequence_length=max_seq_len_for_padding)

print(f"Prompt: '{prompt}'")
print(f"Generated Text: '{generated_output}'")

Prompt: 'machine learning'
Generated Text: 'machine learning the is mouse  '
